# Etapa 1 — Exploração e preparação dos dados

## Projeto Integrador de Ciência de Dados

Este notebook apresenta a primeira etapa do projeto de análise de dados dos hotéis fictícios da cidade de Port Alder, considerando informações referentes aos anos de 2027 e 2028.

O objetivo desta etapa é conhecer a estrutura dos dados, verificar sua integridade, identificar possíveis problemas e preparar as tabelas para as análises estatísticas posteriores.

### Objetivos da exploração

- Carregar e conhecer as sete tabelas disponibilizadas.
- Verificar dimensões, colunas e tipos de dados.
- Identificar e investigar valores ausentes.
- Compreender a granularidade de cada tabela.
- Identificar listas, datas e informações que conectam as tabelas.
- Verificar possíveis chaves primárias e relacionamentos.
- Criar uma função reutilizável para auxiliar na inspeção dos dados.

**Observação:** a base de dados é sintética. Portanto, os resultados e as conclusões deste projeto se referem ao cenário fictício de Port Alder.

## 1. Verificação do ambiente de desenvolvimento

Antes de iniciar a exploração dos dados, foi realizada uma verificação do ambiente de desenvolvimento utilizado no projeto.

Essa etapa permite confirmar a versão do Python, identificar o interpretador em execução e verificar se as bibliotecas necessárias estão instaladas.

Foram verificadas as bibliotecas Pandas, NumPy, Matplotlib, Seaborn, SciPy e Plotly, utilizadas nas etapas de manipulação, análise estatística e visualização dos dados.


In [1]:
import sys
import platform
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

print("=== Interpretador Python ===")
print("Versão do Python :", platform.python_version())
print("Executável       :", sys.executable)
print("Dentro de venv?  :", sys.prefix != sys.base_prefix)

print()
print("=== Bibliotecas instaladas ===")
bibliotecas = ["pandas", "numpy", "matplotlib", "seaborn", "scipy", "plotly", "ipykernel"]
for nome in bibliotecas:
    try:
        print(f"{nome:<12}: {version(nome)}")
    except PackageNotFoundError:
        print(f"{nome:<12}: NÃO INSTALADA")

=== Interpretador Python ===
Versão do Python : 3.15.2
Executável       : c:\Users\luiza\Documents\Projetos\projeto-integrador-ciencia-de-dados\.venv\Scripts\python.exe
Dentro de venv?  : True

=== Bibliotecas instaladas ===
pandas      : 3.1.1
numpy       : 2.5.5
matplotlib  : 3.11.2
seaborn     : 0.13.2
scipy       : 1.18.1
plotly      : 7.1.0
ipykernel   : 7.4.0


## 2. Localização e identificação dos arquivos

Nesta etapa, foi definida a localização da pasta que contém os arquivos disponibilizados para o projeto.

Utilizando a biblioteca `pathlib`, foi possível identificar o caminho da pasta de dados, verificar sua existência e listar os arquivos no formato JSONL.

Também foi realizada uma conferência entre os arquivos encontrados e os sete arquivos esperados, conforme a documentação do conjunto de dados.

Essa verificação garante que todos os arquivos necessários estejam disponíveis antes do carregamento e da análise das tabelas.

In [2]:
from pathlib import Path

PASTA_DADOS = Path("..") / "dados"

print("Pasta de trabalho atual :", Path.cwd())
print("Caminho relativo        :", PASTA_DADOS)
print("Caminho absoluto        :", PASTA_DADOS.resolve())
print("A pasta existe?         :", PASTA_DADOS.exists())

print()
print("=== Arquivos .jsonl encontrados ===")
arquivos_jsonl = sorted(PASTA_DADOS.glob("*.jsonl"))

for arquivo in arquivos_jsonl:
    tamanho_kb = arquivo.stat().st_size / 1024
    print(f"{arquivo.name:<28} {tamanho_kb:>10,.1f} KB")

print("Total encontrado:", len(arquivos_jsonl))

print()
print("=== Conferência com o README ===")

esperados = [
    "hotels.jsonl",
    "room_types.jsonl",
    "daily_market.jsonl",
    "weekly_performance.jsonl",
    "rate_decisions.jsonl",
    "reviews.jsonl",
    "review_scenarios.jsonl",
]

faltando = []

for nome in esperados:
    if not (PASTA_DADOS / nome).is_file():
        faltando.append(nome)

print("Arquivos esperados que estão faltando:", faltando)

Pasta de trabalho atual : c:\Users\luiza\Documents\Projetos\projeto-integrador-ciencia-de-dados\notebooks
Caminho relativo        : ..\dados
Caminho absoluto        : C:\Users\luiza\Documents\Projetos\projeto-integrador-ciencia-de-dados\dados
A pasta existe?         : True

=== Arquivos .jsonl encontrados ===
daily_market.jsonl             11,852.3 KB
hotels.jsonl                        5.6 KB
rate_decisions.jsonl              800.8 KB
review_scenarios.jsonl              2.7 KB
reviews.jsonl                   8,647.5 KB
room_types.jsonl                    7.7 KB
weekly_performance.jsonl        1,217.0 KB
Total encontrado: 7

=== Conferência com o README ===
Arquivos esperados que estão faltando: []


## 3. Carregamento das tabelas

Após confirmar a disponibilidade dos arquivos, iniciou-se o carregamento dos dados utilizando a biblioteca Pandas.

Os arquivos estão no formato JSONL (JSON Lines), em que cada linha representa um registro independente. Para realizar a leitura, foi utilizado o método `pd.read_json()`, com o parâmetro `lines=True`.

Inicialmente, foi carregada a tabela `hotels`, que contém informações gerais dos hotéis. Em seguida, foram carregadas as outras seis tabelas, formando o conjunto de dados utilizado nas análises.

As tabelas foram armazenadas em DataFrames, estrutura que permite organizar, consultar e manipular os dados com Pandas.

In [3]:
# Carregamento do primeiro arquivo JSONL

hotels = pd.read_json(PASTA_DADOS / "hotels.jsonl", lines=True)

print("=== Dimensões da tabela ===")
print("Linhas e colunas:", hotels.shape)

print()
print("=== Primeiros registros ===")
display(hotels.head())

print()
print("=== Nomes das colunas ===")
print(hotels.columns.tolist())

print()
print("=== Tipos de dados ===")
print(hotels.dtypes)

print()
print("=== Informações gerais ===")
hotels.info()

print()
print("=== Valores ausentes ===")
print(hotels.isna().sum())


=== Dimensões da tabela ===
Linhas e colunas: (14, 17)

=== Primeiros registros ===


,hotel_id,name,brand,tier,rooms,opened_year,distance_to_water_km,neighbourhood,comp_set,positioning_index,base_rate,amenities,city,review_count,lifetime_rating,share_rating_5,share_rating_1_2
0,H01,Alder & Quay Hotel,Independent,upscale,168,1998,0.4,Harbour Quay,upscale,1.10,208,"[harbour view, restaurant, gym, parking, pet f...",Port Alder,1204,3.510,0.3131,0.2326
1,H02,The Lantern House,Independent,boutique,54,2016,0.9,Old Harbour,boutique,1.18,236,"[design rooms, rooftop bar, breakfast, bike hire]",Port Alder,420,3.450,0.2929,0.2405
2,H03,Marisol Bay Resort,Solstice Collection,luxury,212,2004,3.1,Marisol Bay,luxury,1.22,352,"[private beach, spa, two restaurants, pool, va...",Port Alder,1538,3.549,0.3121,0.2263
3,H04,Harborlight Suites,Harborlight,upscale,140,2011,1.4,Harbour Quay,upscale,1.06,224,"[all suites, kitchenettes, gym, laundry]",Port Alder,1008,3.325,0.2431,0.2887
4,H05,The Copper Meridian,Independent,luxury,96,2009,2.2,Lighthouse Point,luxury,1.15,318,"[spa, harbour terrace, fine dining, concierge]",Port Alder,677,3.569,0.3235,0.2112



=== Nomes das colunas ===
['hotel_id', 'name', 'brand', 'tier', 'rooms', 'opened_year', 'distance_to_water_km', 'neighbourhood', 'comp_set', 'positioning_index', 'base_rate', 'amenities', 'city', 'review_count', 'lifetime_rating', 'share_rating_5', 'share_rating_1_2']

=== Tipos de dados ===
hotel_id                    str
name                        str
brand                       str
tier                        str
rooms                     int64
opened_year               int64
distance_to_water_km    float64
neighbourhood               str
comp_set                    str
positioning_index       float64
base_rate                 int64
amenities                object
city                        str
review_count              int64
lifetime_rating         float64
share_rating_5          float64
share_rating_1_2        float64
dtype: object

=== Informações gerais ===
<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 17 columns):
 #   Column                N

In [4]:
print("DIMENSÕES:")
print(hotels.shape)

print("\nVALORES AUSENTES:")
print(hotels.isna().sum().to_string())

print("\nPRIMEIROS REGISTROS:")
print(hotels.head().to_string())

DIMENSÕES:
(14, 17)

VALORES AUSENTES:
hotel_id                0
name                    0
brand                   0
tier                    0
rooms                   0
opened_year             0
distance_to_water_km    0
neighbourhood           0
comp_set                0
positioning_index       0
base_rate               0
amenities               0
city                    0
review_count            0
lifetime_rating         0
share_rating_5          0
share_rating_1_2        0

PRIMEIROS REGISTROS:
  hotel_id                 name                brand      tier  rooms  opened_year  distance_to_water_km     neighbourhood  comp_set  positioning_index  base_rate                                               amenities        city  review_count  lifetime_rating  share_rating_5  share_rating_1_2
0      H01   Alder & Quay Hotel          Independent   upscale    168         1998                   0.4      Harbour Quay   upscale               1.10        208  [harbour view, restaurant, gym, parki

## 4. Inspeção inicial das tabelas

Com os arquivos carregados, foi realizada uma inspeção inicial para conhecer a estrutura de cada tabela.

Foram observados:

- Quantidade de linhas e colunas;
- Primeiros registros;
- Nomes das colunas;
- Tipos de dados;
- Quantidade de valores ausentes.

As dimensões encontradas foram comparadas com as informações disponibilizadas no README do conjunto de dados.

Essa comparação permite identificar possíveis divergências entre os arquivos recebidos e a documentação original.

In [5]:
room_types = pd.read_json(PASTA_DADOS / "room_types.jsonl", lines=True)
daily_market = pd.read_json(PASTA_DADOS / "daily_market.jsonl", lines=True)
weekly_performance = pd.read_json(PASTA_DADOS / "weekly_performance.jsonl", lines=True)
rate_decisions = pd.read_json(PASTA_DADOS / "rate_decisions.jsonl", lines=True)
reviews = pd.read_json(PASTA_DADOS / "reviews.jsonl", lines=True)
review_scenarios = pd.read_json(PASTA_DADOS / "review_scenarios.jsonl", lines=True)

tabelas = {
    "room_types": room_types,
    "daily_market": daily_market,
    "weekly_performance": weekly_performance,
    "rate_decisions": rate_decisions,
    "reviews": reviews,
    "review_scenarios": review_scenarios,
}

linhas_esperadas = {
    "room_types": 42,
    "daily_market": 10192,
    "weekly_performance": 1456,
    "rate_decisions": 1442,
    "reviews": 12077,
    "review_scenarios": 11,
}

divergentes = []

for nome, df in tabelas.items():
    print("=" * 70)
    print(f"TABELA: {nome}")
    print("=" * 70)

    print("\n1) Primeiros registros (transposto: colunas nas linhas)")
    display(df.head().T)

    linhas, colunas = df.shape
    print(f"\n2) Dimensões: {linhas:,} linhas x {colunas} colunas")

    esperado = linhas_esperadas[nome]
    confere = linhas == esperado
    print(f"   Linhas esperadas (README): {esperado:,} | obtidas: {linhas:,} | confere? {confere}")
    if not confere:
        divergentes.append(nome)

    print("\n3) Tipos de dados")
    display(df.dtypes.to_frame(name="dtype"))

    print("\n4) Valores ausentes (apenas colunas com pelo menos 1)")
    ausentes = df.isna().sum()
    ausentes = ausentes[ausentes > 0]
    if ausentes.empty:
        print("   Nenhuma coluna com valores ausentes.")
    else:
        resumo_ausentes = pd.DataFrame({
            "ausentes": ausentes,
            "percentual": (ausentes / linhas * 100).round(2),
        })
        display(resumo_ausentes)
        print(f"   Colunas com ausentes: {len(ausentes)} de {colunas}")
    print()

print("Tabelas com número de linhas diferente do README:", divergentes)

TABELA: room_types

1) Primeiros registros (transposto: colunas nas linhas)


,0,1,2,3,4
hotel_id,H01,H01,H01,H02,H02
room_type_id,H01-1,H01-2,H01-3,H02-1,H02-2
room_type,Standard King,Deluxe Harbour,Corner Suite,Petite Double,Loft King
rooms,98,50,20,23,22
max_occupancy,2,2,3,2,2
size_sqm,22,36,50,22,36
rate_premium_pct,0,26,62,0,22
bar_rate,208.0,262.08,336.96,236.0,287.92
view,harbour,courtyard,city,harbour,courtyard
tier,upscale,upscale,upscale,boutique,boutique



2) Dimensões: 42 linhas x 10 colunas
   Linhas esperadas (README): 42 | obtidas: 42 | confere? True

3) Tipos de dados


,dtype
hotel_id,str
room_type_id,str
room_type,str
rooms,int64
max_occupancy,int64
size_sqm,int64
rate_premium_pct,int64
bar_rate,float64
view,str
tier,str



4) Valores ausentes (apenas colunas com pelo menos 1)
   Nenhuma coluna com valores ausentes.

TABELA: daily_market

1) Primeiros registros (transposto: colunas nas linhas)


,0,1,2,3,4
hotel_id,H01,H02,H03,H04,H05
hotel_name,Alder & Quay Hotel,The Lantern House,Marisol Bay Resort,Harborlight Suites,The Copper Meridian
tier,upscale,boutique,luxury,upscale,luxury
comp_set,upscale,boutique,luxury,upscale,luxury
date,2027-01-04 00:00:00,2027-01-04 00:00:00,2027-01-04 00:00:00,2027-01-04 00:00:00,2027-01-04 00:00:00
weekday,Monday,Monday,Monday,Monday,Monday
month,1,1,1,1,1
year,2027,2027,2027,2027,2027
season,winter,winter,winter,winter,winter
demand_event,,,,,


... (19 outputs left out)

,0,1,2,3,4
scenario_id,delight,location,breakfast,service_recovery,room_condition
label,Stay exceeded expectations,Location and access,Breakfast and food,Service recovery,Room condition
share_of_reviews,0.15,0.1,0.09,0.1,0.11
rating_band,"5, 4","5, 4","5, 4, 3","5, 4, 3","3, 2, 1"
trigger,"anticipatory service, a quiet upgrade or a per...","walkable waterfront, ferry or old town access","breakfast quality, choice or service pace",a problem raised during the stay and fixed well,"wear, maintenance or an unfinished room"
expected_sentiment,positive,positive,mixed,positive,negative
recovery_action,thank the guest and record the staff member,none needed,invite the guest back and review the buffet line,log the recovery and brief the shift,block the room and raise a work order



2) Dimensões: 11 linhas x 7 colunas
   Linhas esperadas (README): 11 | obtidas: 11 | confere? True

3) Tipos de dados


,dtype
scenario_id,str
label,str
share_of_reviews,float64
rating_band,str
trigger,str
expected_sentiment,str
recovery_action,str



4) Valores ausentes (apenas colunas com pelo menos 1)
   Nenhuma coluna com valores ausentes.

Tabelas com número de linhas diferente do README: []


In [6]:
print("=== RESUMO DA VALIDAÇÃO DAS TABELAS ===\n")

for nome, df in tabelas.items():
    print(f"\nTABELA: {nome}")
    print("-" * 45)

    print(f"Dimensões: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

    esperado = linhas_esperadas[nome]
    print(f"Linhas esperadas: {esperado:,}")
    print(f"Conferência: {'CORRETO' if df.shape[0] == esperado else 'DIVERGENTE'}")

    ausentes = df.isna().sum()
    total_ausentes = ausentes.sum()

    print(f"Total de valores ausentes: {total_ausentes:,}")

    if total_ausentes > 0:
        print("Colunas com valores ausentes:")
        print(ausentes[ausentes > 0].to_string())
    else:
        print("Nenhum valor ausente.")

print("\n=== FIM DA VALIDAÇÃO ===")

=== RESUMO DA VALIDAÇÃO DAS TABELAS ===


TABELA: room_types
---------------------------------------------
Dimensões: 42 linhas x 10 colunas
Linhas esperadas: 42
Conferência: CORRETO
Total de valores ausentes: 0
Nenhum valor ausente.

TABELA: daily_market
---------------------------------------------
Dimensões: 10,192 linhas x 56 colunas
Linhas esperadas: 10,192
Conferência: CORRETO
Total de valores ausentes: 0
Nenhum valor ausente.

TABELA: weekly_performance
---------------------------------------------
Dimensões: 1,456 linhas x 41 colunas
Linhas esperadas: 1,456
Conferência: CORRETO
Total de valores ausentes: 836
Colunas com valores ausentes:
flow_through       728
mean_rating         27
positive_share      27
sentiment_index     27
response_rate       27

TABELA: rate_decisions
---------------------------------------------
Dimensões: 1,442 linhas x 24 colunas
Linhas esperadas: 1,442
Conferência: CORRETO
Total de valores ausentes: 0
Nenhum valor ausente.

TABELA: reviews
-----------

In [7]:
resumo = []

for nome, df in tabelas.items():
    resumo.append({
        "tabela": nome,
        "linhas": df.shape[0],
        "colunas": df.shape[1],
        "linhas_corretas": df.shape[0] == linhas_esperadas[nome],
        "valores_ausentes": df.isna().sum().sum(),
        "colunas_com_ausentes": (df.isna().sum() > 0).sum()
    })

resumo_df = pd.DataFrame(resumo)

display(resumo_df)

,tabela,linhas,colunas,linhas_corretas,valores_ausentes,colunas_com_ausentes
0,room_types,42,10,True,0,0
1,daily_market,10192,56,True,0,0
2,weekly_performance,1456,41,True,836,5
3,rate_decisions,1442,24,True,0,0
4,reviews,12077,26,True,3769,1
5,review_scenarios,11,7,True,0,0


In [8]:
for nome in ["weekly_performance", "reviews"]:
    print(f"\n=== {nome} ===")
    df = tabelas[nome]
    
    ausentes = df.isna().sum()
    print(ausentes[ausentes > 0])
    


=== weekly_performance ===
flow_through       728
mean_rating         27
positive_share      27
sentiment_index     27
response_rate       27
dtype: int64

=== reviews ===
response_hours    3769
dtype: int64


## 5. Criação de uma função reutilizável

Para padronizar a inspeção das tabelas e evitar a repetição de comandos, foi desenvolvida a função `inspecionar_tabela()`.

A função recebe um DataFrame, seu nome e, opcionalmente, a quantidade de linhas esperada conforme o README.

Ela apresenta as dimensões, os tipos de dados, os valores ausentes e os textos vazios, além de retornar um dicionário com os principais resultados.

Posteriormente, a função foi reutilizada nas sete tabelas, facilitando a comparação e a identificação de possíveis inconsistências.

In [10]:
def inspecionar_tabela(df, nome, linhas_esperadas=None):
    """Mostra dimensões, tipos, valores ausentes e textos vazios de uma tabela, sem modificá-la.

    Parâmetros:
        df: o DataFrame a inspecionar.
        nome: nome da tabela, usado apenas nos títulos da saída.
        linhas_esperadas: número de linhas indicado no README (opcional).

    Retorna:
        Um dicionário com os números principais da inspeção.
    """
    print("=" * 70)
    print(f"TABELA: {nome}")
    print("=" * 70)

    linhas, colunas = df.shape
    print(f"Dimensões: {linhas:,} linhas x {colunas} colunas")

    confere = None
    if linhas_esperadas is not None:
        confere = linhas == linhas_esperadas
        print(f"Linhas esperadas (README): {linhas_esperadas:,} | obtidas: {linhas:,} | confere? {confere}")

    print("\nTipos de dados")
    display(df.dtypes.to_frame(name="dtype"))

    ausentes = df.isna().sum()
    ausentes = ausentes[ausentes > 0]
    print("\nValores ausentes (NaN/None/NaT) - apenas colunas com pelo menos 1")
    if ausentes.empty:
        print("Nenhuma coluna com valores ausentes.")
    else:
        resumo_ausentes = pd.DataFrame({
            "ausentes": ausentes,
            "percentual": (ausentes / linhas * 100).round(2),
        })
        display(resumo_ausentes)

    textos_vazios = {}
    for coluna in df.columns:
        if pd.api.types.is_string_dtype(df[coluna]):
            quantidade = int((df[coluna] == "").sum())
            if quantidade > 0:
                textos_vazios[coluna] = quantidade

    print('\nTextos vazios ("") - apenas colunas de texto com pelo menos 1')
    if not textos_vazios:
        print("Nenhuma coluna de texto com valores vazios.")
    else:
        vazios = pd.Series(textos_vazios, name="vazios")
        resumo_vazios = pd.DataFrame({
            "vazios": vazios,
            "percentual": (vazios / linhas * 100).round(2),
        })
        display(resumo_vazios)
    print()

    return {
        "tabela": nome,
        "linhas": linhas,
        "colunas": colunas,
        "linhas_esperadas": linhas_esperadas,
        "confere": confere,
        "colunas_com_ausentes": len(ausentes),
        "total_ausentes": int(ausentes.sum()),
        "colunas_com_texto_vazio": len(textos_vazios),
        "total_textos_vazios": sum(textos_vazios.values()),
    }

In [14]:
resultados = []

resultados.append(inspecionar_tabela(hotels, "hotels", linhas_esperadas=14))
resultados.append(inspecionar_tabela(room_types, "room_types", linhas_esperadas=42))
resultados.append(inspecionar_tabela(review_scenarios, "review_scenarios", linhas_esperadas=11))

print("Resumo devolvido pela função:")

for resultado in resultados:
    print(resultado)
    

TABELA: hotels
Dimensões: 14 linhas x 17 colunas
Linhas esperadas (README): 14 | obtidas: 14 | confere? True

Tipos de dados


,dtype
hotel_id,str
name,str
brand,str
tier,str
rooms,int64
opened_year,int64
distance_to_water_km,float64
neighbourhood,str
comp_set,str
positioning_index,float64



Valores ausentes (NaN/None/NaT) - apenas colunas com pelo menos 1
Nenhuma coluna com valores ausentes.

Textos vazios ("") - apenas colunas de texto com pelo menos 1
Nenhuma coluna de texto com valores vazios.

TABELA: room_types
Dimensões: 42 linhas x 10 colunas
Linhas esperadas (README): 42 | obtidas: 42 | confere? True

Tipos de dados


,dtype
hotel_id,str
room_type_id,str
room_type,str
rooms,int64
max_occupancy,int64
size_sqm,int64
rate_premium_pct,int64
bar_rate,float64
view,str
tier,str



Valores ausentes (NaN/None/NaT) - apenas colunas com pelo menos 1
Nenhuma coluna com valores ausentes.

Textos vazios ("") - apenas colunas de texto com pelo menos 1
Nenhuma coluna de texto com valores vazios.

TABELA: review_scenarios
Dimensões: 11 linhas x 7 colunas
Linhas esperadas (README): 11 | obtidas: 11 | confere? True

Tipos de dados


,dtype
scenario_id,str
label,str
share_of_reviews,float64
rating_band,str
trigger,str
expected_sentiment,str
recovery_action,str



Valores ausentes (NaN/None/NaT) - apenas colunas com pelo menos 1
Nenhuma coluna com valores ausentes.

Textos vazios ("") - apenas colunas de texto com pelo menos 1
Nenhuma coluna de texto com valores vazios.

Resumo devolvido pela função:
{'tabela': 'hotels', 'linhas': 14, 'colunas': 17, 'linhas_esperadas': 14, 'confere': True, 'colunas_com_ausentes': 0, 'total_ausentes': 0, 'colunas_com_texto_vazio': 0, 'total_textos_vazios': 0}
{'tabela': 'room_types', 'linhas': 42, 'colunas': 10, 'linhas_esperadas': 42, 'confere': True, 'colunas_com_ausentes': 0, 'total_ausentes': 0, 'colunas_com_texto_vazio': 0, 'total_textos_vazios': 0}
{'tabela': 'review_scenarios', 'linhas': 11, 'colunas': 7, 'linhas_esperadas': 11, 'confere': True, 'colunas_com_ausentes': 0, 'total_ausentes': 0, 'colunas_com_texto_vazio': 0, 'total_textos_vazios': 0}


## 6. Granularidade das tabelas

A granularidade representa o nível de detalhe de cada registro de uma tabela. Compreender esse aspecto é essencial para evitar interpretações incorretas e combinações inadequadas entre dados.

Durante a exploração, foram identificadas diferentes granularidades:

- `hotels`: um registro por hotel;
- `room_types`: um registro por tipo de quarto de um hotel;
- `daily_market`: um registro por hotel em uma noite;
- `weekly_performance`: um registro por hotel em uma semana;
- `rate_decisions`: uma decisão de preço de um hotel em uma semana;
- `reviews`: um registro por avaliação de hóspede;
- `review_scenarios`: um registro por cenário de avaliação.

Essa distinção será importante nas próximas etapas, especialmente nas análises temporais e nos relacionamentos entre tabelas.

In [15]:
tabelas_todas = {
    "hotels": hotels,
    "room_types": room_types,
    "daily_market": daily_market,
    "weekly_performance": weekly_performance,
    "rate_decisions": rate_decisions,
    "reviews": reviews,
    "review_scenarios": review_scenarios,
}

linhas_esperadas_todas = {
    "hotels": 14,
    "room_types": 42,
    "daily_market": 10192,
    "weekly_performance": 1456,
    "rate_decisions": 1442,
    "reviews": 12077,
    "review_scenarios": 11,
}

print("Tabelas carregadas:", len(tabelas_todas))
print("Nomes:", list(tabelas_todas.keys()))

Tabelas carregadas: 7
Nomes: ['hotels', 'room_types', 'daily_market', 'weekly_performance', 'rate_decisions', 'reviews', 'review_scenarios']


In [16]:
resultados_todas = []
for nome, df in tabelas_todas.items():
    resultado = inspecionar_tabela(df, nome, linhas_esperadas=linhas_esperadas_todas[nome])
    resultados_todas.append(resultado)

TABELA: hotels
Dimensões: 14 linhas x 17 colunas
Linhas esperadas (README): 14 | obtidas: 14 | confere? True

Tipos de dados


,dtype
hotel_id,str
name,str
brand,str
tier,str
rooms,int64
opened_year,int64
distance_to_water_km,float64
neighbourhood,str
comp_set,str
positioning_index,float64



Valores ausentes (NaN/None/NaT) - apenas colunas com pelo menos 1
Nenhuma coluna com valores ausentes.

Textos vazios ("") - apenas colunas de texto com pelo menos 1
Nenhuma coluna de texto com valores vazios.

TABELA: room_types
Dimensões: 42 linhas x 10 colunas
Linhas esperadas (README): 42 | obtidas: 42 | confere? True

Tipos de dados


,dtype
hotel_id,str
room_type_id,str
room_type,str
rooms,int64
max_occupancy,int64
size_sqm,int64
rate_premium_pct,int64
bar_rate,float64
view,str
tier,str



Valores ausentes (NaN/None/NaT) - apenas colunas com pelo menos 1
Nenhuma coluna com valores ausentes.

Textos vazios ("") - apenas colunas de texto com pelo menos 1
Nenhuma coluna de texto com valores vazios.

TABELA: daily_market
Dimensões: 10,192 linhas x 56 colunas
Linhas esperadas (README): 10,192 | obtidas: 10,192 | confere? True

Tipos de dados


,dtype
hotel_id,str
hotel_name,str
tier,str
comp_set,str
date,datetime64[us]
weekday,str
month,int64
year,int64
season,str
demand_event,str


... (17 outputs left out)

,vazios,percentual
response_text,3769,31.21



TABELA: review_scenarios
Dimensões: 11 linhas x 7 colunas
Linhas esperadas (README): 11 | obtidas: 11 | confere? True

Tipos de dados


,dtype
scenario_id,str
label,str
share_of_reviews,float64
rating_band,str
trigger,str
expected_sentiment,str
recovery_action,str



Valores ausentes (NaN/None/NaT) - apenas colunas com pelo menos 1
Nenhuma coluna com valores ausentes.

Textos vazios ("") - apenas colunas de texto com pelo menos 1
Nenhuma coluna de texto com valores vazios.



In [17]:
nao_conferem = []
for resultado in resultados_todas:
    if resultado["confere"] is not True:
        nao_conferem.append(resultado["tabela"])
print("Tabelas que não conferem com o README:", nao_conferem)

quadro_resumo = pd.DataFrame(resultados_todas)
display(quadro_resumo)

Tabelas que não conferem com o README: []


,tabela,linhas,colunas,linhas_esperadas,confere,colunas_com_ausentes,total_ausentes,colunas_com_texto_vazio,total_textos_vazios
0,hotels,14,17,14,True,0,0,0,0
1,room_types,42,10,42,True,0,0,0,0
2,daily_market,10192,56,10192,True,0,0,1,9534
3,weekly_performance,1456,41,1456,True,5,836,1,1246
4,rate_decisions,1442,24,1442,True,0,0,1,1246
5,reviews,12077,26,12077,True,1,3769,1,3769
6,review_scenarios,11,7,11,True,0,0,0,0


In [18]:
granularidade = {
    "hotels": "um hotel",
    "room_types": "um tipo de quarto de um hotel",
    "daily_market": "um hotel em uma noite",
    "weekly_performance": "um hotel em uma semana (começa na segunda-feira)",
    "rate_decisions": "uma decisão de preço de um hotel em uma semana, com a semana seguinte anexada",
    "reviews": "uma avaliação de hóspede",
    "review_scenarios": "um cenário de avaliação",
}

quadro_resumo["granularidade"] = quadro_resumo["tabela"].map(granularidade)
display(quadro_resumo[["tabela", "linhas", "colunas", "granularidade"]])


,tabela,linhas,colunas,granularidade
0,hotels,14,17,um hotel
1,room_types,42,10,um tipo de quarto de um hotel
2,daily_market,10192,56,um hotel em uma noite
3,weekly_performance,1456,41,um hotel em uma semana (começa na segunda-feira)
4,rate_decisions,1442,24,uma decisão de preço de um hotel em uma semana...
5,reviews,12077,26,uma avaliação de hóspede
6,review_scenarios,11,7,um cenário de avaliação


## 7. Identificação de datas e estruturas especiais

Durante a análise das colunas, foram identificados campos que exigiam atenção especial.

Primeiramente, foram verificadas as colunas relacionadas a datas, considerando os diferentes períodos de registro das tabelas. Esses campos foram convertidos para o tipo `datetime`, permitindo operações temporais mais adequadas.

Também foi analisada a coluna `amenities`, presente na tabela `hotels`. A inspeção demonstrou que seus valores já estavam armazenados como listas, permitindo consultar diretamente os serviços oferecidos pelos hotéis.

In [20]:
# Reunindo as sete tabelas para inspeção
tabelas_com_hotels = {
    "hotels": hotels,
    **tabelas
}

# Identificando possíveis colunas de datas
termos_data = [
    "date", "time", "week", "day",
    "created", "updated", "period"
]

for nome, df in tabelas_com_hotels.items():
    print("=" * 70)
    print(f"TABELA: {nome}")
    print("=" * 70)

    print("\nColunas existentes:")
    print(df.columns.tolist())

    colunas_data = [
        coluna for coluna in df.columns
        if any(termo in coluna.lower() for termo in termos_data)
    ]

    print("\nPossíveis colunas de datas:")

    if colunas_data:
        for coluna in colunas_data:
            print(f"\nColuna: {coluna}")
            print(f"Tipo: {df[coluna].dtype}")
            print("Exemplos:")
            print(df[coluna].head(3).tolist())
    else:
        print("Nenhuma identificada pelos nomes.")

TABELA: hotels

Colunas existentes:
['hotel_id', 'name', 'brand', 'tier', 'rooms', 'opened_year', 'distance_to_water_km', 'neighbourhood', 'comp_set', 'positioning_index', 'base_rate', 'amenities', 'city', 'review_count', 'lifetime_rating', 'share_rating_5', 'share_rating_1_2']

Possíveis colunas de datas:

Coluna: lifetime_rating
Tipo: float64
Exemplos:
[3.51, 3.45, 3.549]
TABELA: room_types

Colunas existentes:
['hotel_id', 'room_type_id', 'room_type', 'rooms', 'max_occupancy', 'size_sqm', 'rate_premium_pct', 'bar_rate', 'view', 'tier']

Possíveis colunas de datas:
Nenhuma identificada pelos nomes.
TABELA: daily_market

Colunas existentes:
['hotel_id', 'hotel_name', 'tier', 'comp_set', 'date', 'weekday', 'month', 'year', 'season', 'demand_event', 'rooms_available', 'rooms_sold', 'occupancy', 'adr', 'room_revenue', 'ancillary_revenue', 'commission', 'net_revenue', 'gross_operating_profit', 'bookings', 'cancellations', 'cancellation_rate', 'booking_window_days', 'length_of_stay', 'avai

In [21]:
print("TIPOS DE DADOS EM AMENITIES:")

print(
    hotels["amenities"]
    .map(lambda valor: type(valor).__name__)
    .value_counts()
)

print("\nPRIMEIROS CINCO VALORES:")

for valor in hotels["amenities"].head(5):
    print(repr(valor))
    

TIPOS DE DADOS EM AMENITIES:
amenities
list    14
Name: count, dtype: int64

PRIMEIROS CINCO VALORES:
['harbour view', 'restaurant', 'gym', 'parking', 'pet friendly']
['design rooms', 'rooftop bar', 'breakfast', 'bike hire']
['private beach', 'spa', 'two restaurants', 'pool', 'valet']
['all suites', 'kitchenettes', 'gym', 'laundry']
['spa', 'harbour terrace', 'fine dining', 'concierge']


In [23]:
for nome in ["rate_decisions", "reviews"]:
    print(f"\nTABELA: {nome}")
    print(tabelas[nome].columns.tolist())
    


TABELA: rate_decisions
['decision_id', 'hotel_id', 'week_start', 'action', 'current_adr', 'mars_rate', 'rate_position', 'rate_gap_pct', 'booking_pace_index', 'forecast_occupancy', 'suggested_rate', 'suggested_change_pct', 'rate_guardrail', 'expected_revpar', 'expected_lift_pct', 'confidence', 'rationale', 'cancellation_rate', 'demand_event', 'realised_occupancy', 'realised_adr', 'realised_revpar', 'realised_uplift_pct', 'outcome']

TABELA: reviews
['review_id', 'hotel_id', 'review_date', 'stay_date', 'week_start', 'scenario_id', 'rating', 'title', 'text', 'channel', 'segment', 'nights', 'room_type', 'verified_stay', 'cleanliness', 'comfort', 'service_score', 'location_score', 'value_score', 'sleep_quality', 'would_return', 'sentiment', 'sentiment_score', 'lexicon_version', 'response_text', 'response_hours']


In [42]:
colunas_datas = {
    "daily_market": ["date"],
    "weekly_performance": ["week_start"],
    "rate_decisions": ["week_start"],
    "reviews": ["review_date", "stay_date", "week_start"]
}


In [27]:
for nome_tabela, colunas in colunas_datas.items():
    df = tabelas[nome_tabela]

    for coluna in colunas:
        df[coluna] = pd.to_datetime(
            df[coluna],
            errors="coerce"
        )

        print(f"\n{nome_tabela}.{coluna}")
        print("Tipo:", df[coluna].dtype)
        print("Datas inválidas:", df[coluna].isna().sum())
        print("Exemplos:", df[coluna].head(3).tolist())
        print("-" * 50)
        


daily_market.date
Tipo: datetime64[us]
Datas inválidas: 0
Exemplos: [Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-04 00:00:00')]
--------------------------------------------------

weekly_performance.week_start
Tipo: datetime64[us]
Datas inválidas: 0
Exemplos: [Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-11 00:00:00'), Timestamp('2027-01-18 00:00:00')]
--------------------------------------------------

rate_decisions.week_start
Tipo: datetime64[us]
Datas inválidas: 0
Exemplos: [Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-04 00:00:00')]
--------------------------------------------------

reviews.review_date
Tipo: datetime64[us]
Datas inválidas: 0
Exemplos: [Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-05 00:00:00')]
--------------------------------------------------

reviews.stay_date
Tipo: datetime64[us]
Datas inválidas: 0
Exemplos: [Timestamp('2027-

In [43]:
for coluna in ["stay_date", "week_start"]:
    print(f"\nreviews.{coluna}")
    print("Tipo:", reviews[coluna].dtype)
    print("Datas inválidas:", reviews[coluna].isna().sum())


reviews.stay_date
Tipo: datetime64[us]
Datas inválidas: 0

reviews.week_start
Tipo: datetime64[us]
Datas inválidas: 0



review_date
Tipo: datetime64[us]
Valores inválidos: 0
Exemplos: [Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-05 00:00:00')]

stay_date
Tipo: datetime64[us]
Valores inválidos: 0
Exemplos: [Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-04 00:00:00')]

week_start
Tipo: datetime64[us]
Valores inválidos: 0
Exemplos: [Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-04 00:00:00'), Timestamp('2027-01-04 00:00:00')]


In [30]:
quantidade_restaurantes = hotels["amenities"].apply(
    lambda lista: any(
        "restaurant" in item.lower()
        for item in lista
    )
).sum()

print("Hotéis com restaurante:", quantidade_restaurantes)

Hotéis com restaurante: 5


In [31]:
for nome, df in tabelas_com_hotels.items():
    print(f"\n{'=' * 50}")
    print(f"TABELA: {nome}")
    print("=" * 50)

    print(df.columns.tolist())
    


TABELA: hotels
['hotel_id', 'name', 'brand', 'tier', 'rooms', 'opened_year', 'distance_to_water_km', 'neighbourhood', 'comp_set', 'positioning_index', 'base_rate', 'amenities', 'city', 'review_count', 'lifetime_rating', 'share_rating_5', 'share_rating_1_2']

TABELA: room_types
['hotel_id', 'room_type_id', 'room_type', 'rooms', 'max_occupancy', 'size_sqm', 'rate_premium_pct', 'bar_rate', 'view', 'tier']

TABELA: daily_market
['hotel_id', 'hotel_name', 'tier', 'comp_set', 'date', 'weekday', 'month', 'year', 'season', 'demand_event', 'rooms_available', 'rooms_sold', 'occupancy', 'adr', 'room_revenue', 'ancillary_revenue', 'commission', 'net_revenue', 'gross_operating_profit', 'bookings', 'cancellations', 'cancellation_rate', 'booking_window_days', 'length_of_stay', 'available_rooms', 'revpar', 'trevpar', 'revpor', 'net_revpar', 'goppar', 'cpor', 'ancillary_capture', 'market_occupancy', 'market_adr', 'demand_index', 'otb_occupancy_index', 'comp_set_occupancy', 'comp_set_adr', 'comp_set_re

## 8. Identificação de chaves e relacionamentos

Nesta etapa, foram identificadas possíveis chaves primárias e estrangeiras para compreender como as tabelas se relacionam.

As chaves primárias permitem identificar registros de forma única, enquanto as chaves estrangeiras possibilitam estabelecer conexões entre diferentes tabelas.

Foram verificadas as correspondências dos identificadores dos hotéis, dos cenários de avaliação e dos tipos de quarto.

Também foram realizadas verificações de duplicidade e de valores ausentes nas possíveis chaves primárias, buscando garantir a integridade das informações.

In [32]:
# Possíveis chaves primárias das tabelas

chaves = {
    "hotels": ["hotel_id"],
    "room_types": ["hotel_id", "room_type_id"],
    "daily_market": ["hotel_id", "date"],
    "weekly_performance": ["hotel_id", "week_start"],
    "rate_decisions": ["decision_id"],
    "reviews": ["review_id"],
    "review_scenarios": ["scenario_id"]
}

for nome, colunas in chaves.items():
    df = tabelas_com_hotels[nome]

    duplicados = df.duplicated(subset=colunas).sum()

    print(f"\nTabela: {nome}")
    print(f"Chave analisada: {colunas}")
    print(f"Registros duplicados: {duplicados}")
    print(f"Chave única? {duplicados == 0}")


Tabela: hotels
Chave analisada: ['hotel_id']
Registros duplicados: 0
Chave única? True

Tabela: room_types
Chave analisada: ['hotel_id', 'room_type_id']
Registros duplicados: 0
Chave única? True

Tabela: daily_market
Chave analisada: ['hotel_id', 'date']
Registros duplicados: 0
Chave única? True

Tabela: weekly_performance
Chave analisada: ['hotel_id', 'week_start']
Registros duplicados: 0
Chave única? True

Tabela: rate_decisions
Chave analisada: ['decision_id']
Registros duplicados: 0
Chave única? True

Tabela: reviews
Chave analisada: ['review_id']
Registros duplicados: 0
Chave única? True

Tabela: review_scenarios
Chave analisada: ['scenario_id']
Registros duplicados: 0
Chave única? True


In [33]:
# Verificar se os hotel_id existem na tabela hotels

for nome in ["room_types", "daily_market", "weekly_performance", "rate_decisions", "reviews"]:

    df = tabelas_com_hotels[nome]

    inexistentes = (~df["hotel_id"].isin(hotels["hotel_id"])).sum()

    print(f"\nTabela: {nome}")
    print(f"hotel_id sem correspondência: {inexistentes}")


# Verificar scenario_id das avaliações

cenarios_inexistentes = (
    ~reviews["scenario_id"].isin(review_scenarios["scenario_id"])
).sum()

print("\nTabela: reviews")
print(f"scenario_id sem correspondência: {cenarios_inexistentes}")


Tabela: room_types
hotel_id sem correspondência: 0

Tabela: daily_market
hotel_id sem correspondência: 0

Tabela: weekly_performance
hotel_id sem correspondência: 0

Tabela: rate_decisions
hotel_id sem correspondência: 0

Tabela: reviews
hotel_id sem correspondência: 0

Tabela: reviews
scenario_id sem correspondência: 0


In [34]:
tipos_avaliacoes = reviews[["hotel_id", "room_type"]].drop_duplicates()

tipos_quartos = room_types[["hotel_id", "room_type"]].drop_duplicates()

comparacao = tipos_avaliacoes.merge(
    tipos_quartos,
    on=["hotel_id", "room_type"],
    how="left",
    indicator=True
)

print(comparacao["_merge"].value_counts())


_merge
both          42
left_only      0
right_only     0
Name: count, dtype: int64


## 9. Investigação dos valores ausentes e textos vazios

Após identificar os valores ausentes nas tabelas, foi realizada uma investigação para compreender suas possíveis causas.

A análise considerou especialmente três situações:

- A coluna `flow_through`, na tabela `weekly_performance`;
- As métricas de avaliações, como `mean_rating`, `positive_share`, `sentiment_index` e `response_rate`;
- Os campos `response_hours` e `response_text`, na tabela `reviews`.

Também foram examinados os campos `demand_event`, verificando a diferença entre valores nulos e textos vazios.

Essa distinção é importante porque um campo contendo `""` não é considerado ausente pelo método `isna()` do Pandas.

Nenhum valor foi removido ou preenchido durante essa investigação. A intenção foi compreender a origem das ausências antes de tomar qualquer decisão sobre o tratamento dos dados.

In [35]:
print("Verificação do ambiente")

print("Pandas:", pd.__version__)
print("hotels:", hotels.shape)
print("weekly_performance:", weekly_performance.shape)
print("reviews:", reviews.shape)

print("\nColunas necessárias:")
print(weekly_performance[
    ["hotel_id", "week_start", "flow_through", "review_count",
     "mean_rating", "positive_share", "sentiment_index", "response_rate"]
].head())

Verificação do ambiente
Pandas: 3.1.1
hotels: (14, 17)
weekly_performance: (1456, 41)
reviews: (12077, 26)

Colunas necessárias:
  hotel_id week_start  flow_through  review_count  mean_rating  \
0      H01 2027-01-04           NaN             8        4.250   
1      H01 2027-01-11           NaN             9        3.889   
2      H01 2027-01-18           NaN             5        2.600   
3      H01 2027-01-25           NaN             8        3.875   
4      H01 2027-02-01           NaN             8        2.250   

   positive_share  sentiment_index  response_rate  
0          0.7500           0.2365         0.5000  
1          0.5556           0.2452         0.5556  
2          0.0000          -0.4753         1.0000  
3          0.7500           0.0667         0.5000  
4          0.2500          -0.2416         1.0000  


In [36]:
# Organizar os registros por hotel e semana
weekly_ordenado = weekly_performance.sort_values(
    ["hotel_id", "week_start"]
)

# Investigar flow_through
primeiras_52 = weekly_ordenado.groupby("hotel_id").head(52)

print("=== FLOW_THROUGH ===")
print("Ausentes nas primeiras 52 semanas:",
      primeiras_52["flow_through"].isna().sum())

print("Ausentes no total:",
      weekly_performance["flow_through"].isna().sum())

print("\nAusentes por hotel:")
print(
    weekly_performance.groupby("hotel_id")["flow_through"]
    .apply(lambda coluna: coluna.isna().sum())
)

# Investigar métricas de avaliação
colunas_avaliacao = [
    "mean_rating",
    "positive_share",
    "sentiment_index",
    "response_rate"
]

print("\n=== MÉTRICAS DE AVALIAÇÃO ===")

for coluna in colunas_avaliacao:
    print(f"{coluna}: {weekly_performance[coluna].isna().sum()}")

# Verificar se as quatro métricas estão ausentes nas mesmas linhas
ausentes_todos = weekly_performance[colunas_avaliacao].isna().all(axis=1)

print("\nRegistros com as quatro métricas ausentes:",
      ausentes_todos.sum())

print("\nReview_count nesses registros:")
print(
    weekly_performance.loc[
        ausentes_todos, "review_count"
    ].value_counts(dropna=False)
)

=== FLOW_THROUGH ===
Ausentes nas primeiras 52 semanas: 728
Ausentes no total: 728

Ausentes por hotel:
hotel_id
H01    52
H02    52
H03    52
H04    52
H05    52
H06    52
H07    52
H08    52
H09    52
H10    52
H11    52
H12    52
H13    52
H14    52
Name: flow_through, dtype: int64

=== MÉTRICAS DE AVALIAÇÃO ===
mean_rating: 27
positive_share: 27
sentiment_index: 27
response_rate: 27

Registros com as quatro métricas ausentes: 27

Review_count nesses registros:
review_count
0    27
Name: count, dtype: int64


In [37]:
# ==========================================
# INVESTIGAÇÃO DA TABELA REVIEWS
# ==========================================

print("========== RESPONSE_HOURS ==========")

print("Valores ausentes:", reviews["response_hours"].isna().sum())

print("\nDistribuição dos valores de resposta:")

print(
    reviews["response_hours"].isna().value_counts()
)


print("\n========== RESPONSE_TEXT ==========")

print("Valores nulos:", reviews["response_text"].isna().sum())

print(
    "Textos vazios:",
    (reviews["response_text"] == "").sum()
)


print("\n========== RELAÇÃO ENTRE OS CAMPOS ==========")

sem_horas = reviews["response_hours"].isna()
sem_texto = reviews["response_text"].eq("")

print(
    "Sem horas de resposta:",
    sem_horas.sum()
)

print(
    "Sem texto de resposta:",
    sem_texto.sum()
)

print(
    "Sem horas e sem texto:",
    (sem_horas & sem_texto).sum()
)

print(
    "Sem horas, mas com texto:",
    (sem_horas & ~sem_texto & reviews["response_text"].notna()).sum()
)

print(
    "Com horas, mas sem texto:",
    (reviews["response_hours"].notna() & sem_texto).sum()
)


========== RESPONSE_HOURS ==========
Valores ausentes: 3769

Distribuição dos valores de resposta:
response_hours
False    8308
True     3769
Name: count, dtype: int64

========== RESPONSE_TEXT ==========
Valores nulos: 0
Textos vazios: 3769

========== RELAÇÃO ENTRE OS CAMPOS ==========
Sem horas de resposta: 3769
Sem texto de resposta: 3769
Sem horas e sem texto: 3769
Sem horas, mas com texto: 0
Com horas, mas sem texto: 0


In [38]:
# ==========================================
# INVESTIGAÇÃO DE DEMAND_EVENT
# ==========================================

tabelas_eventos = {
    "daily_market": daily_market,
    "weekly_performance": weekly_performance,
    "rate_decisions": rate_decisions
}

for nome, df in tabelas_eventos.items():

    print(f"\n========== {nome.upper()} ==========")

    print("Valores nulos:", df["demand_event"].isna().sum())

    print(
        "Textos vazios:",
        df["demand_event"].eq("").sum()
    )

    print(
        "Eventos preenchidos:",
        (df["demand_event"].notna() & df["demand_event"].ne("")).sum()
    )
    


========== DAILY_MARKET ==========
Valores nulos: 0
Textos vazios: 9534
Eventos preenchidos: 658

========== WEEKLY_PERFORMANCE ==========
Valores nulos: 0
Textos vazios: 1246
Eventos preenchidos: 210

========== RATE_DECISIONS ==========
Valores nulos: 0
Textos vazios: 1246
Eventos preenchidos: 196


In [39]:
# Verificar textos vazios ou compostos apenas por espaços

for nome, df, colunas in [
    ("daily_market", daily_market, ["demand_event"]),
    ("weekly_performance", weekly_performance, ["demand_event"]),
    ("rate_decisions", rate_decisions, ["demand_event"]),
    ("reviews", reviews, ["response_text"])
]:

    print(f"\n{nome.upper()}")

    for coluna in colunas:
        apenas_espacos = df[coluna].apply(
            lambda valor: isinstance(valor, str)
            and valor != ""
            and valor.strip() == ""
        ).sum()

        print(f"{coluna}: {apenas_espacos} textos com apenas espaços")


DAILY_MARKET
demand_event: 0 textos com apenas espaços

WEEKLY_PERFORMANCE
demand_event: 0 textos com apenas espaços

RATE_DECISIONS
demand_event: 0 textos com apenas espaços

REVIEWS
response_text: 0 textos com apenas espaços


In [40]:


chaves_primarias = {
    "hotels": ["hotel_id"],
    "room_types": ["hotel_id", "room_type_id"],
    "daily_market": ["hotel_id", "date"],
    "weekly_performance": ["hotel_id", "week_start"],
    "rate_decisions": ["decision_id"],
    "reviews": ["review_id"],
    "review_scenarios": ["scenario_id"]
}

for nome, colunas in chaves_primarias.items():

    df = tabelas_com_hotels[nome]

    duplicados = df.duplicated(subset=colunas).sum()
    ausentes = df[colunas].isna().any(axis=1).sum()

    print(f"\nTabela: {nome}")
    print(f"Duplicações: {duplicados}")
    print(f"Chaves com valores ausentes: {ausentes}")
    print(f"Chave válida: {duplicados == 0 and ausentes == 0}")


Tabela: hotels
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: room_types
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: daily_market
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: weekly_performance
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: rate_decisions
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: reviews
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: review_scenarios
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True


## 10. Conclusão da exploração dos dados

A exploração permitiu conhecer a estrutura das sete tabelas, verificar suas dimensões, identificar os tipos de dados, compreender suas granularidades e investigar possíveis problemas de qualidade.

As quantidades de registros encontradas correspondem às informações apresentadas no README.

Também foram identificadas as principais relações entre as tabelas e verificadas as possíveis chaves primárias, sem duplicações ou valores ausentes nas chaves analisadas.

A investigação dos valores ausentes demonstrou que diferentes situações possuem justificativas específicas, não sendo adequado realizar preenchimentos ou exclusões sem considerar o contexto de cada campo.

Com essa análise inicial, os dados estão mais bem compreendidos e preparados para as próximas etapas do projeto.

In [41]:
# AUDITORIA FINAL DAS CHAVES PRIMÁRIAS

chaves_primarias = {
    "hotels": ["hotel_id"],
    "room_types": ["hotel_id", "room_type_id"],
    "daily_market": ["hotel_id", "date"],
    "weekly_performance": ["hotel_id", "week_start"],
    "rate_decisions": ["decision_id"],
    "reviews": ["review_id"],
    "review_scenarios": ["scenario_id"]
}

for nome, colunas in chaves_primarias.items():

    df = tabelas_com_hotels[nome]

    duplicados = df.duplicated(subset=colunas).sum()
    ausentes = df[colunas].isna().any(axis=1).sum()

    print(f"\nTabela: {nome}")
    print(f"Duplicações: {duplicados}")
    print(f"Chaves com valores ausentes: {ausentes}")
    print(f"Chave válida: {duplicados == 0 and ausentes == 0}")


Tabela: hotels
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: room_types
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: daily_market
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: weekly_performance
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: rate_decisions
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: reviews
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True

Tabela: review_scenarios
Duplicações: 0
Chaves com valores ausentes: 0
Chave válida: True
